# Bay Area Microclimate Weather Prediction

**Goal:** Build an ML model that predicts localized weather conditions across distinct microclimate zones in the San Francisco Bay Area.

**Approach:** Collect weather data from multiple sources (Synoptic stations, Open-Meteo dense grid, ERA5 reanalysis), train a model with spatial features as inputs, then cluster on model-derived weather profiles to define microclimate zones.

**S3 Bucket:** `s3://bay-area-microclimate/`

---

## Project Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                        DATA SOURCES                             │
├───────────────┬──────────────────┬──────────────────────────────┤
│  Synoptic API │  Open-Meteo Dense│  ERA5 (Open-Meteo)           │
│  ~1,124 stns  │  ~900 grid pts   │  36 grid pts (0.25°)         │
│  1-yr history │  0.05° (~5km)    │  20-yr history available      │
│  (ground truth│  20-yr available │  (synoptic context)           │
│   obs)        │  (no key needed) │                              │
└──────┬────────┴────────┬─────────┴──────────────┬──────────────┘
       │                 │                        │
       ▼                 ▼                        ▼
┌─────────────────────────────────────────────────────────────────┐
│                     S3: raw/ (parquet)                           │
│  synoptic/monthly/YYYY-MM/chunk_XXXX.parquet                    │
│  openmeteo_dense/monthly/YYYY-MM/grid_{lat}_{lon}.parquet       │
│  era5/monthly/YYYY-MM/grid_{lat}_{lon}.parquet                  │
└──────────────────────────┬──────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────┐
│                   MODEL-FIRST WORKFLOW                           │
│  1. Train model with spatial features as inputs                 │
│     (elevation, coastal dist, terrain exposure, etc.)           │
│  2. Extract learned representations / predicted profiles        │
│  3. Cluster on predicted weather behavior → microclimate zones  │
└──────────────────────────┬──────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────┐
│                 S3: features/ (parquet)                          │
│  static/stations_with_features.parquet                          │
│  zones/zone_assignments.parquet  (model-derived)                │
│  zones/zone_profiles.parquet     (model-derived)                │
└─────────────────────────────────────────────────────────────────┘
```

## Pipeline Status Dashboard

| Component | Script | Status | Blocker |
|-----------|--------|--------|---------|
| Synoptic download | `src/download_synoptic.py` | **Verified** (3/299 chunks) | Full 1-yr run not completed |
| ERA5 download | `src/download_era5.py` | Implemented | Not yet run |
| Open-Meteo dense grid | `src/download_openmeteo_dense.py` | Implemented | Not yet run |
| Static features | `src/compute_static_features.py` | Implemented | Needs Synoptic metadata in S3 |
| Zone clustering (legacy) | `src/cluster_zones.py` | Implemented | Superseded by model-first workflow |
| Spatial-feature model | — | Not implemented | Needs all data sources |
| Model-derived zones | — | Not implemented | Needs trained model |

*Weather Underground pipeline (`src/download_wunderground.py`) was abandoned — WU API access is unreliable. Replaced by Open-Meteo dense grid.*

---

## Data Sources Detail

### 1. Synoptic API (`src/download_synoptic.py`)

Surface weather observations from professional and amateur stations.

- **Stations:** ~1,124 in Bay Area bbox + 19 known ASOS stations (KSFO, KOAK, KSJC, etc.)
- **Variables:** `temp_c`, `humidity`, `wind_speed_kph`, `wind_dir_deg`, `precip_mm`
- **Chunking:** 50 stations per API call (~180k rows/month/chunk)
- **S3 layout:** `raw/synoptic/monthly/YYYY-MM/chunk_XXXX.parquet`
- **Full run:** 13 months x 23 chunks = 299 API calls
- **Free tier limit:** 1 year of history (hard cap — blocks the 20-year goal)

**Verified results (first run):**
| Chunk | Rows |
|-------|------|
| `chunk_0000` | 155,218 |
| `chunk_0001` | 182,085 |
| `chunk_0002` | 186,401 |

**Key gotchas:**
- Use `SYNOPTIC_TOKEN` (generated token), not the API key directly — API key returns 401
- Variable name is `precip_accum`, NOT `precip_accumulated` (wrong name returns 0 stations)
- No client-side RESTRICTED filter needed — API returns only accessible data

### 2. Open-Meteo Dense Grid (`src/download_openmeteo_dense.py`)

High-resolution gridded historical weather data — replaces Weather Underground.

- **Grid:** ~900 points at 0.05° spacing (~5km) across Bay Area bbox
- **API:** Open-Meteo Archive (same as ERA5 source, no API key needed)
- **History:** 20+ years available (back to 1940)
- **Variables:** `temp_c`, `humidity`, `wind_speed_kph`, `wind_dir_deg`, `precip_mm`, `cloud_cover_pct`, `pressure_hpa`
- **S3 layout:** `raw/openmeteo_dense/monthly/YYYY-MM/grid_{lat}_{lon}.parquet`
- **Rate limiting:** 1s sleep between calls

**Why Open-Meteo over Weather Underground:**
- No API key or PWS device registration required
- Consistent data quality (model-interpolated, no noisy backyard sensors)
- Deterministic coverage (no station discovery, no gaps)
- 20+ year history (partially offsets Synoptic's 1-year free tier cap)

**Tradeoff:** Model output, not direct observations. Won't capture hyper-local effects (street-level heat islands, building shadows). Synoptic ASOS stations provide ground-truth calibration.

### 3. ERA5 Reanalysis (`src/download_era5.py`)

Large-scale atmospheric context via Open-Meteo Archive API.

- **Grid:** 6x6 = 36 points at 0.25° resolution (~27km)
- **History:** Back to 1940 (no free-tier limit — offsets Synoptic's 1-yr cap)
- **Variables:**
  - Temperature (2m surface + 850hPa free-atmosphere)
  - Relative humidity, surface pressure, wind speed/direction
  - Precipitation, cloud cover
  - **Boundary layer height** (key for marine layer modeling)
- **S3 layout:** `raw/era5/monthly/YYYY-MM/grid_{lat}_{lon}.parquet`
- **Rate limiting:** 1s sleep between calls, no API key needed

**Design rationale:** ERA5 is intentionally coarse — it provides synoptic context (what is the large-scale atmosphere doing?). Microclimate signal comes from station observations.

---

## Feature Engineering & Modeling Strategy

### Static Features (`src/compute_static_features.py`)

Geographic features computed per station, independent of weather:

| Feature | Description | Source |
|---------|-------------|--------|
| `dist_coast_km` | Great-circle distance to Pacific coast waypoints | Computed (manual waypoints) |
| `dist_bay_km` | Distance to SF Bay shoreline waypoints | Computed (manual waypoints) |
| `elev_m` | Station elevation | Synoptic metadata (feet, converted) |
| `coastal_exposure` | Composite heuristic: `1 - dist_coast_norm - elev_norm * 0.3` | Derived |

**Known limitations:**
- Coast/bay distances use manually defined waypoints (sufficient for ~10km zones, imprecise in complex geometry like Sausalito/Tiburon)
- Synoptic elevation can be inaccurate; SRTM 30m DEM would be ground truth
- No terrain aspect (slope/aspect) for cold-air pooling and shadow effects
- `coastal_exposure` is hand-crafted, not learned — to be replaced by model

### Model-First Zone Definition (revised 2026-03-10)

Instead of pre-clustering stations into zones, the workflow is now **model-first, cluster-second:**

1. **Train model with spatial features as inputs** — elevation, coastal distance, bay distance, slope aspect, terrain exposure, etc. The model learns how spatial features relate to weather outcomes.
2. **Extract learned representations** — pull internal embeddings or evaluate predicted weather behavior across a spatial grid (predicted fog frequency, temperature variance, diurnal patterns).
3. **Cluster on predicted profiles** — zones emerge from model-derived weather behavior, not raw geography. Sunset and Mission naturally separate because their predicted profiles differ.

**Why this replaces pre-clustering:**
- No need to choose K upfront
- Not constrained by hand-crafted features like `coastal_exposure`
- Zones adapt automatically as data sources are added
- `src/cluster_zones.py` retained for baseline comparison

---

## Unified Schema

### Synoptic observations

| Column | Type | Notes |
|--------|------|-------|
| `datetime` | timestamp | UTC |
| `temp_c` | float | Celsius |
| `humidity` | float | Relative humidity % |
| `wind_speed_kph` | float | km/h |
| `wind_dir_deg` | float | Degrees (0-360) |
| `precip_mm` | float | Accumulated precipitation, mm |
| `stid` | string | Station identifier |
| `name` | string | Station name |
| `lat` | float | Latitude |
| `lon` | float | Longitude |
| `elev_m` | float | **Always meters** (Synoptic converts from feet via `* 0.3048`) |
| `network` | string | Station network |
| `source` | string | `"synoptic"` |

### Open-Meteo dense grid

| Column | Type | Notes |
|--------|------|-------|
| `datetime` | timestamp | UTC |
| `temp_c` | float | Celsius |
| `humidity` | float | Relative humidity % |
| `wind_speed_kph` | float | km/h |
| `wind_dir_deg` | float | Degrees (0-360) |
| `precip_mm` | float | Hourly precipitation, mm |
| `cloud_cover_pct` | float | Total cloud cover % |
| `pressure_hpa` | float | Surface pressure, hPa |
| `grid_lat` | float | Grid point latitude |
| `grid_lon` | float | Grid point longitude |
| `source` | string | `"openmeteo_dense"` |

---

## Secrets & Credentials

| Secret | Env Var | Status |
|--------|---------|--------|
| Synoptic API token | `SYNOPTIC_TOKEN` | Active (rotated after exposure incident) |
| AWS credentials | `~/.aws/credentials` | Active (boto3 auto-managed) |

*Open-Meteo and ERA5 downloads require no API key.*

**Security notes:**
- All secrets via env vars, never hardcoded
- `.gitignore` excludes `*.env`, `*accessKeys.csv`
- Both Synoptic token and AWS creds were rotated after a past terminal exposure incident

---

## Dependencies

| Package | Purpose |
|---------|---------|
| `pandas` | Data manipulation |
| `boto3` / `botocore` | S3 upload/download |
| `pyarrow` | Parquet serialization |
| `requests` | API calls |
| `python-dotenv` | Load `.env` files |
| `scikit-learn` | K-means clustering |
| `matplotlib` | Elbow plots |
| `dateutil` | Date arithmetic |

---

## Open Items & Next Steps

### Immediate (unblocked)
- [ ] Complete Synoptic 1-year download (all 13 months x 23 chunks in S3, verify KSFO/KOAK rows present)
- [ ] Run ERA5 download (all 36 grid points x 13 months in `raw/era5/monthly/`)
- [ ] Run Open-Meteo dense grid download (~900 grid points x 13 months in `raw/openmeteo_dense/monthly/`)
- [ ] Run static features once Synoptic metadata is in S3

### Modeling workflow (after data downloads)
- [ ] Define train/val/test split strategy (temporal, spatial, or stratified)
- [ ] Train spatial-feature model (elevation, coastal distance, terrain exposure as inputs)
- [ ] Extract learned representations / predicted weather profiles across spatial grid
- [ ] Cluster on predicted profiles to define microclimate zones
- [ ] Compare model-derived zones against legacy K-means baseline (`cluster_zones.py`)

### Design decisions needed
- [ ] **20-year data strategy** — Open-Meteo dense grid can go back 20 years freely. Synoptic is capped at 1 year. Decide whether to download extended Open-Meteo history and use Synoptic as calibration only.
- [ ] **Validate Open-Meteo dense grid vs. Synoptic ASOS** — Compute bias/RMSE at co-located points (KSFO, KOAK, KSJC) to quantify model vs. observation differences.